## 1. Import the libraries:

In [35]:
import pandas as pd
import os
import numpy as np
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
os.chdir("../")
%pwd

'D:\\ML and AI Engineer\\Projects\\An-end-to-end-customer-churn-prediction-model'

## 2. Reading the training data:

In [17]:
train = pd.read_parquet("Artifacts/data_validation/valid/train.parquet")
train.head(2)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,537472,22594,CHRISTMAS GINGHAM TREE,24,2010-12-07 11:11:00,0.85,12971,United Kingdom
1,532756,22984,"CARD, GINGHAM ROSE",12,2010-11-14 13:39:00,0.42,14769,United Kingdom


In [18]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 410693 entries, 0 to 410692
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      410693 non-null  object        
 1   StockCode    410693 non-null  object        
 2   Description  410693 non-null  object        
 3   Quantity     410693 non-null  int64         
 4   InvoiceDate  410693 non-null  datetime64[ns]
 5   Price        410693 non-null  float64       
 6   Customer ID  410693 non-null  int64         
 7   Country      410693 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 25.1+ MB


## 3. Data Cleaning:

In [19]:
train.nunique()

Invoice        27324
StockCode       4076
Description     4529
Quantity         474
InvoiceDate    25420
Price            657
Customer ID     4749
Country           40
dtype: int64

In [20]:
train.shape

(410693, 8)

### 3.1 Check for 0s:

In [27]:
(train==0.0).sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

#### Replace 0s with `np.nan` for correct statistical calculations:

In [23]:
train['Price'] = train['Price'].replace(0.0, np.nan)

In [25]:
(train==0).sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

In [29]:
train.isna().sum()

Invoice         0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
Price          31
Customer ID     0
Country         0
dtype: int64

### Imputing the missing or `np.nan` values:

In [36]:
# Create a price imputer:
price_imputer = SimpleImputer(strategy='median')
price_imputer.fit(train[['Price']])

# Transform:
train['Price'] = price_imputer.transform(train[['Price']])

In [38]:
train.isna().sum()

Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
dtype: int64

### 3.2 Check for duplicates:

In [39]:
duplicates = train.duplicated()
print(duplicates.any(), "\n")

True 



In [40]:
print(f"There is a total of: {train[duplicates].shape[0]:,} duplicate rows. Original data has: {train.shape[0]:,} rows.")

There is a total of: 14,530 duplicate rows. Original data has: 410,693 rows.


Dropping the duplicates:

In [41]:
train.drop_duplicates(inplace=True)
print(f"Number of rows after dropping the duplicates: {train.shape[0]:,}")

Number of rows after dropping the duplicates: 396,163


## 3.2 Handling Returned Quantity & Cancelled Orders:

In [42]:
train.head(2)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,537472,22594,CHRISTMAS GINGHAM TREE,24,2010-12-07 11:11:00,0.85,12971,United Kingdom
1,532756,22984,"CARD, GINGHAM ROSE",12,2010-11-14 13:39:00,0.42,14769,United Kingdom


In [44]:
train['orders_returned'] = [1 if q < 0  else 0 for q in train['Quantity']]
print(f"A total of {train.value_counts('orders_returned').iloc[1]:,} orders were returned.")

A total of 9,588 orders were returned.


In [45]:
train['orders_cancelled'] = [1 if invoice.startswith('C') else 0 for invoice in train['Invoice']]
print(f"A total of {train.value_counts('orders_cancelled').iloc[1]:,} cancelled orders were found.")

A total of 9,588 cancelled orders were found.


#### Create a new column quantities to abs value:

In [46]:
train['Quantity_abs'] = train['Quantity'].map(lambda q: np.abs(q))

## 3.3 Outlier Detection:

In [47]:
numerical_cols = ['Quantity_abs', 'Price']   

In [49]:
for col in numerical_cols:
    Q1 = np.percentile(train[col], 25)
    Q3 = np.percentile(train[col], 75)
    IQR = Q3 - Q1
    multiplier = 1.5
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    
    print(f"The lower bound for {col} is: {lower_bound}")
    print(f"The upper bound for {col} is: {upper_bound}")
    outliers_below_lower_bound = train[train[col] <= lower_bound]
    print(f"The number of transactions flagged as outliers below the lower bound ({lower_bound}) are: {len(train[train[col] <= lower_bound]):,}")
    print(f"The number of transactions flagged as outliers above the upper bound ({upper_bound}) are: {len(train[train[col]>= upper_bound]):,}")
    print(f"{'-' * 50}")
    

The lower bound for Quantity_abs is: -13.0
The upper bound for Quantity_abs is: 27.0
The number of transactions flagged as outliers below the lower bound (-13.0) are: 0
The number of transactions flagged as outliers above the upper bound (27.0) are: 26,064
--------------------------------------------------
The lower bound for Price is: -2.5
The upper bound for Price is: 7.5
The number of transactions flagged as outliers below the lower bound (-2.5) are: 0
The number of transactions flagged as outliers above the upper bound (7.5) are: 34,774
--------------------------------------------------


In [56]:
print(f"The max number of ordered quantity is: {train['Quantity'].max():,}")
print(f"The max number of returned quantity is: {train['Quantity'].min():,}")
print(f"The minimum number of returned quantity is: {train[train['Quantity'] < 0].max()['Quantity']:,}")

The max number of ordered quantity is: 74,215
The max number of returned quantity is: -74,215
The minimum number of returned quantity is: -1


In [55]:
print(f"The maximum price is: {train['Price'].max():,}")
print(f"The minimum price is: {train['Price'].min():,}")


The maximum price is: 25,111.09
The minimum price is: 0.001
